# 10 - Onchain Metrics

Active addresses, gas, staking.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Load

In [2]:
try:
    oc = sql("SELECT metric, ts, value FROM onchain ORDER BY ts")
    oc["date"] = pd.to_datetime(oc.ts, unit="ms")
    print("metrics:", oc.metric.unique().tolist())
    print("rows:", len(oc))
except Exception as e:
    print("onchain empty:", e)
    oc = pd.DataFrame()

metrics: ['active_addresses', 'gas_price_gwei', 'eth_staked']
rows: 3


## 2. Latest values

In [3]:
if not oc.empty:
    latest = oc.sort_values("ts").groupby("metric").tail(1)
    print(latest[["metric","value"]])

             metric       value
0  active_addresses    512000.0
1    gas_price_gwei        14.3
2        eth_staked  34200000.0


## 3. Action

In [4]:
if not oc.empty and "eth_staked" in oc.metric.values:
    staked = oc[oc.metric=="eth_staked"].sort_values("ts")
    chg = (staked.value.iloc[-1] / staked.value.iloc[-30] - 1)*100 if len(staked)>30 else 0
    if chg > 1:   action, reason = "LONG", "staking inflow rising"
    else:         action, reason = "FLAT", "staking flat"
else:
    action, reason = "FLAT", "no onchain signal available"
action_block("ONCHAIN", action, reason)

  ONCHAIN
  time   : 2026-10-04 10:47 UTC
  ACTION : FLAT
  REASON : staking flat


{'action': 'FLAT',
 'reason': 'staking flat',
 'price': None,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}